In [9]:
import pandas as pd
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import re
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader
import os
import zipfile

# ==========================================
# 1. CẤU HÌNH CƠ BẢN VÀ CHUẨN BỊ DATA
# ==========================================
MODEL_NAME = "../model/marbert_base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

STANCE2ID = {"Against": 0, "Favor": 1, "None": 2}
ID2LABEL = {0: "Against", 1: "Favor", 2: "None"}
SENTIMENT2ID = {"Negative": 0, "Neutral": 1, "Positive": 2}
SARCASM2ID = {"No": 0, "Yes": 1}

def clean_arabic_tweet(text):
    if not isinstance(text, str): return ""
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"\u0640", "", text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"(.)\1+", r"\1\1", text)
    return re.sub(r"\s+", " ", text.replace("#", " ")).strip()

def load_data(file_path, is_train=True):
    df = pd.read_csv(file_path, keep_default_na=False)
    for col in ["target", "text"]:
        df[col] = df[col].astype(str).str.strip()
    df["clean_text"] = df["text"].apply(clean_arabic_tweet)
    
    if is_train:
        df["label_stance"] = df["stance"].map(STANCE2ID).fillna(2).astype(int)
        df["label_sentiment"] = df["sentiment"].map(SENTIMENT2ID).fillna(1).astype(int)
        df["label_sarcasm"] = df["sarcasm"].map(SARCASM2ID).fillna(0).astype(int)
    return df

print("Đang nạp dữ liệu...")
train_df = load_data("../data/train.csv", is_train=True)
dev_df = load_data("../data/dev.csv", is_train=False)

# FIX BUG TOKEN_TYPE_IDS: Truyền dạng cặp câu (target, clean_text) chuẩn Hugging Face
def tokenize_func(examples):
    tokenized = tokenizer(
        examples["target"], 
        examples["clean_text"], 
        padding="max_length", 
        truncation=True, 
        max_length=128
    )
    if "label_stance" in examples:
        tokenized["labels_stance"] = examples["label_stance"]
        tokenized["labels_sentiment"] = examples["label_sentiment"]
        tokenized["labels_sarcasm"] = examples["label_sarcasm"]
    return tokenized

# ==========================================
# 2. KIẾN TRÚC MULTI-TASK & TRAINER (BẢN TỐI ƯU HÓA)
# ==========================================
class MultiTaskMARBERT(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        
        # [FIX 2] THÊM DROPOUT ĐỂ CHỐNG OVERFITTING VÀ TĂNG ĐỒNG NHẤT VỚI ARABERT
        self.dropout = nn.Dropout(0.2)
        
        self.stance_head = nn.Linear(hidden_size, 3)
        self.sentiment_head = nn.Linear(hidden_size, 3)
        self.sarcasm_head = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        # Đi qua Dropout trước khi vào Classification Head
        pooled = self.dropout(outputs.pooler_output) 
        return self.stance_head(pooled), self.sentiment_head(pooled), self.sarcasm_head(pooled)

class MultiTaskTrainer(Trainer):
    # Đã bỏ hàm __init__ chứa stance_weights theo phương án "thuần khiết hóa"
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels_stance = inputs.pop("labels_stance")
        labels_sentiment = inputs.pop("labels_sentiment")
        labels_sarcasm = inputs.pop("labels_sarcasm")
        
        logits_stance, logits_sentiment, logits_sarcasm = model(**inputs)
        
        # TẮT CLASS WEIGHTS: Trở về CE thuần khiết, chỉ giữ Label Smoothing
        loss_fct_stance = nn.CrossEntropyLoss(label_smoothing=0.1)
        loss_fct_sentiment = nn.CrossEntropyLoss()
        loss_fct_sarcasm = nn.CrossEntropyLoss()
        
        # Truyền logit và label tương ứng vào từng hàm loss để lấy ra giá trị tensor trước khi nhân hệ số
        loss_stance = loss_fct_stance(logits_stance, labels_stance)
        loss_sentiment = loss_fct_sentiment(logits_sentiment, labels_sentiment)
        loss_sarcasm = loss_fct_sarcasm(logits_sarcasm, labels_sarcasm)
        
        total_loss = loss_stance + 0.1 * loss_sentiment + 0.05 * loss_sarcasm
        
        return (total_loss, {"logits_stance": logits_stance}) if return_outputs else total_loss

# [FIX 1] BỔ SUNG HÀM TÍNH METRIC ĐỂ TRAINER ĐÁNH GIÁ (Lấy lại từ bản cũ)
def compute_metrics(eval_pred):
    logits_tuple = eval_pred.predictions
    labels_tuple = eval_pred.label_ids

    # Trích xuất logits của Stance
    if isinstance(logits_tuple, (tuple, list)):
        logits = logits_tuple[0] 
    elif isinstance(logits_tuple, dict):
         logits = logits_tuple["logits_stance"]
    else:
        logits = logits_tuple
        
    # Trích xuất labels của Stance
    if isinstance(labels_tuple, (tuple, list)):
        labels = labels_tuple[0]
    else:
        labels = labels_tuple

    logits = np.array(logits)
    labels = np.array(labels)
    
    if logits.ndim > 2: 
         logits = logits.reshape(-1, logits.shape[-1])
         labels = labels.flatten()
    elif logits.ndim == 1 and labels.ndim == 0: 
         logits = logits.reshape(1, -1)
         labels = labels.reshape(1)

    preds = np.argmax(logits, axis=-1)
    
    f_ag = f1_score(labels, preds, labels=[0], average="macro")
    f_fav = f1_score(labels, preds, labels=[1], average="macro")
    
    return {"Favg2": (f_fav + f_ag) / 2.0}

# ==========================================
# 3. QUY TRÌNH 10-FOLD CV & TRÍCH XUẤT XÁC SUẤT OOF
# ==========================================
print("\n--- BẮT ĐẦU 10-FOLD CROSS VALIDATION VỚI MARBERT ---")
N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Khởi tạo mảng trống để chứa toàn bộ xác suất Out-Of-Fold
oof_probs = np.zeros((len(train_df), 3))

# [FIX 3] STRATIFY THEO TỔ HỢP TARGET + STANCE
strat_key = train_df["target"] + "_" + train_df["stance"]

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, strat_key)):
    print(f"\n🚀 ĐANG HUẤN LUYỆN FOLD {fold + 1}/{N_SPLITS}...")
    
    fold_train_df = train_df.iloc[train_idx]
    fold_val_df = train_df.iloc[val_idx]
    
    cols_data = ["target", "clean_text", "label_stance", "label_sentiment", "label_sarcasm"]
    fold_train_ds = Dataset.from_pandas(fold_train_df[cols_data]).map(tokenize_func, batched=True).remove_columns(cols_data)
    fold_val_ds = Dataset.from_pandas(fold_val_df[cols_data]).map(tokenize_func, batched=True).remove_columns(cols_data)
    
    model = MultiTaskMARBERT(MODEL_NAME)
    
    # [FIX 1 & 4] CẬP NHẬT TRAINING ARGUMENTS ĐỂ LƯU BEST MODEL & BẬT BF16
    args = TrainingArguments(
        output_dir=f"../model/marbert_fold_{fold}",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        num_train_epochs=6, 
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        weight_decay=0.01,
        
        # Bật Mixed Precision (Bfloat16 an toàn, không tràn số)
        bf16=True, 
        
        # Chiến thuật lưu mô hình: Theo sát Favg2 trên tập Eval
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="Favg2",
        greater_is_better=True,
        
        label_names=["labels_stance", "labels_sentiment", "labels_sarcasm"],
        logging_steps=100,
    )
    
    trainer = MultiTaskTrainer(
        model=model, 
        args=args, 
        train_dataset=fold_train_ds, 
        eval_dataset=fold_val_ds, # Truyền tập Validation vào đây để Trainer đánh giá
        data_collator=data_collator,
        compute_metrics=compute_metrics, # Truyền hàm tính điểm vào
    )
    
    # Bắt đầu huấn luyện. Khi kết thúc, Trainer sẽ tự động load lại bộ weights của Epoch có Favg2 cao nhất.
    trainer.train()
    
    # Lưu Model Tốt Nhất
    torch.save(model.state_dict(), f"../model/multitask_fold_{fold}.pt")
    
    # Trích xuất OOF Probs cho Fold này
    val_loader = DataLoader(
        fold_val_ds.with_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids"]), 
        batch_size=32, shuffle=False
    )
    
    model.eval()
    model.to(device)
    fold_val_probs = []
    with torch.no_grad():
        for batch in val_loader:
            outputs = model(
                input_ids=batch["input_ids"].to(device), 
                attention_mask=batch["attention_mask"].to(device),
                token_type_ids=batch["token_type_ids"].to(device)
            )
            probs = F.softmax(outputs[0], dim=-1)
            fold_val_probs.append(probs.cpu().numpy())
    
    oof_probs[val_idx] = np.vstack(fold_val_probs)
    
    del model, trainer
    torch.cuda.empty_cache()

# Lưu mảng OOF Probs chuẩn
np.save("../model/marbert_oof_probs.npy", oof_probs)
print("\n✅ Đã trích xuất và lưu thành công tệp: '../model/marbert_oof_probs.npy'")

# ==========================================
# 4. INFERENCE LÊN TẬP TEST VÀ XUẤT XÁC SUẤT TEST
# ==========================================
print("\n--- TIẾN HÀNH INFERENCE LÊN TẬP TEST (DEV) ---")
cols_test = ["target", "clean_text"]
test_ds = Dataset.from_pandas(dev_df[cols_test]).map(tokenize_func, batched=True).remove_columns(cols_test)
test_loader = DataLoader(
    test_ds.with_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids"]), 
    batch_size=32, shuffle=False
)

all_fold_test_probs = []

for fold in range(N_SPLITS):
    print(f"Đang inference tập Test bằng Model Fold {fold + 1}...")
    model = MultiTaskMARBERT(MODEL_NAME).to(device)
    model.load_state_dict(torch.load(f"../model/multitask_fold_{fold}.pt", map_location=device, weights_only=True))
    model.eval()
    
    fold_test_probs = []
    with torch.no_grad():
        for batch in test_loader:
            outputs = model(
                input_ids=batch["input_ids"].to(device), 
                attention_mask=batch["attention_mask"].to(device),
                token_type_ids=batch["token_type_ids"].to(device)
            )
            probs = F.softmax(outputs[0], dim=-1)
            fold_test_probs.append(probs.cpu().numpy())
            
    all_fold_test_probs.append(np.vstack(fold_test_probs))
    del model
    torch.cuda.empty_cache()

# Tính trung bình xác suất (Soft Voting) của 10 nếp gấp mô hình
final_probs = np.mean(all_fold_test_probs, axis=0)
np.save("../model/marbert_test_probs.npy", final_probs)
print("✅ Đã trích xuất và lưu thành công tệp: '../model/marbert_test_probs.npy'")

# Thử dự đoán bằng Argmax mặc định xem hiệu năng của phôi MARBERT nâng cấp ra sao
final_preds = np.argmax(final_probs, axis=-1)
if "stance" in dev_df.columns:
    dev_df["label_stance"] = dev_df["stance"].map(STANCE2ID).fillna(2).astype(int)
    f_ag = f1_score(dev_df["label_stance"].values, final_preds, labels=[0], average="macro")
    f_fav = f1_score(dev_df["label_stance"].values, final_preds, labels=[1], average="macro")
    print(f"📊 Điểm Favg2 ước tính trên Dev (Chưa tune Threshold): {(f_ag + f_fav) / 2.0:.4f}")

Đang nạp dữ liệu...

--- BẮT ĐẦU 10-FOLD CROSS VALIDATION VỚI MARBERT ---

🚀 ĐANG HUẤN LUYỆN FOLD 1/10...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5277.02it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Favg2
1,1.066083,0.777987,0.798885
2,0.719520,0.709500,0.822385
3,0.552499,0.737305,0.843632
4,0.475652,0.768884,0.835066
5,0.419576,0.777743,0.832314
6,0.407172,0.777774,0.828784



🚀 ĐANG HUẤN LUYỆN FOLD 2/10...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3905.99it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Đang inference tập Test bằng Model Fold 9...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2762.68it/s]


Đang inference tập Test bằng Model Fold 10...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4656.20it/s]


✅ Đã trích xuất và lưu thành công tệp: '../model/marbert_test_probs.npy'
📊 Điểm Favg2 ước tính trên Dev (Chưa tune Threshold): 0.8527
